# Transformers Attention Visualization: an exploration of the self-attention mechanism

**TP emprunté ici: https://coling.epfl.ch/TP/TP-DL.html**

This practical sessions was produced using [Jupyter](http://jupyter.org). If you are used to it, you can [download the corresponding notebook code from here](TP-TransformersAttention.ipynb). If not, no problem at all, this is not mandatory: simply proceed as usual in your favorite Python environment.

## Introduction

The aim of this practical session is to get yourself acquainted with the self-attention mechanism of [Transformer](https://arxiv.org/abs/1706.03762) language models (LMs) such as [BERT](https://arxiv.org/abs/1810.04805) and [GPT-2](https://d4mucfpksywv.cloudfront.net/better-language-models/language-models.pdf).

As we have seen in the [last practical session](https://coling.epfl.ch/TP/TP-TextClassification.php) and today's class, recurrent neural networks such as LSTM's can help attend to past tokens by computing the similarity between decoder hidden state and encoder output states. However, the problem with such a computation is that recurrent functions cannot be parallelized because previous state need to be computed to encode next one. Instead with self-attention we compute pairwise scores between each encoder hidden state and the
other encoder hidden states. Today's class will be an exploration of this mechanism and which linguistic functions it may encode.

1. Neuron view exploration:
    - How do the query, key, value calculations look like for BERT and GPT-2?
    - What do each of these matrices learn? Are there any patterns given certain inputs?
    - **Use case:** Linking neurons to model behavior
2. Attention-head view exploration:
    - What do different heads in different layers attend to? What differentiates them?
    - Are there lexical patterns that can be distinctly encoded by each?
    - **Use case:** Detecting model bias
3. Model view exploration:
    - Can we detect which heads do certain behaviors without inspecting them all individiually?
    - **Use case:** Locating relevant attention-heads from a birds-eye view

### ⚠️ **Note: You do not need to execute any cells. It will take a little while to do so. But feel free to explore different sentences as well.** ⚠️

### Acknowledgements

Huge thanks to [Jesse Vig](https://jessevig.com/) for creating the [BertViz package](https://github.com/jessevig/bertviz) that this notebook depends on! We were heavily inspired by the package's [paper](https://aclanthology.org/P19-3007.pdf) and blog ([part 1](https://towardsdatascience.com/deconstructing-bert-distilling-6-patterns-from-100-million-parameters-b49113672f77) and [part 2](https://towardsdatascience.com/deconstructing-bert-part-2-visualizing-the-inner-workings-of-attention-60a16d86b5c1)) to come up with the use-case examples. A big part of the usage text is also copy-pasted from the [interactive colab](https://colab.research.google.com/drive/1hXIQ77A4TYS4y3UthWF-Ci7V7vVUoxmQ?usp=sharing) provided by the package.

---

## Setting up your environment

While you can download the following packages with `pip` to your computer directly, we recommend **(but not require)** you to use a [virtual environment](https://packaging.python.org/en/latest/guides/installing-using-pip-and-virtual-environments/) to not mess up the package versions for different project. If you'd like to, here is a [quick tutorial](https://docs.google.com/document/d/1D8TapyWrfyWijfrGq5BtYfrAoWmSmgcP-Gx3i2MXWvo/edit) on virtual environments that you can checkout with an EPFL email.

Alternatively you can use the EPFL jupyter notebook service [noto](https://noto.epfl.ch/), however you will have to `pip install` some specific packages such as bertviz.

1. First make sure you have (a virtual environment (e.g., [venv, virtualenv](https://docs.python.org/3/library/venv.html), [conda](https://docs.conda.io/en/latest/miniconda.html)), and that the environment has) a Python version >= 3.6, per package requirements. If you are using the a Jupyter Notebook, make sure the interpreter points to the correct `python` executable.

In [ ]:
%pip -q install -U ipykernel
%pip -q install -U ipywidgets
%pip -q install -U pip setuptools wheel
%pip -q install -U bertviz

In [ ]:
%pip -q install -U bertviz

3. Now you should be able to load a language model like BERT from [huggingface](https://huggingface.co/)'s [transformers package](https://huggingface.co/docs/transformers/index) and use for the following sections:

In [ ]:
# 1) Import the relevant packages
from bertviz import head_view, model_view
from transformers import BertTokenizer, BertModel, utils
utils.logging.set_verbosity_error()  # Suppress standard warnings

# 2) Load model and tokenizer
model_version = 'bert-base-uncased'
model = BertModel.from_pretrained(model_version, output_attentions=True)
tokenizer = BertTokenizer.from_pretrained(model_version)

# 3) Tokenize the desired sentences
sentence_a = "The cat sat on the mat."
sentence_b = "The cat lay on the rug."
inputs = tokenizer.encode_plus(sentence_a, sentence_b, return_tensors='pt')
input_ids = inputs['input_ids']
token_type_ids = inputs['token_type_ids']

# 4) Get the attention by passing the inputs to the model
attention = model(input_ids, token_type_ids=token_type_ids)[-1]
sentence_b_start = token_type_ids[0].tolist().index(1)
input_id_list = input_ids[0].tolist() # Batch index 0
tokens = tokenizer.convert_ids_to_tokens(input_id_list)

## 1) Neuron View Exploration
The neuron view visualizes the intermediate representations (e.g. query and key vectors) that are used to compute attention.

### a) Instructions on how to use the neuron view
In this introductory section, we take a look at a popular model trained with the masked language modeling objective: **BERT**.

You can watch [this video](https://vimeo.com/358488181?embedded=true&source=vimeo_logo&owner=99152109) to understand how to use the neuron view. In the collapsed view (initial state), the lines show the attention from each token (left) to every other token (right). In the expanded view, the tool traces the chain of computations that produce these attention weights. For a more detailed explanation, you can refer to the [second blog post](https://towardsdatascience.com/deconstructing-bert-part-2-visualizing-the-inner-workings-of-attention-60a16d86b5c1) made by Jesse Vig.

👉 **Hover** over any of the tokens on the left side of the visualization to filter attention from that token.<br/>
👉 Then **click** on the **plus** icon that is revealed **when hovering over the tokens on the left**. **This exposes the query vectors, key vectors, and other intermediate representations** used to compute the attention weights. Each color band represents a single neuron value, where color intensity indicates the magnitude and hue the sign (blue=positive, orange=negative).<br/>
👉 Once in the expanded view, **hover** over any other **token** on the left to see the associated attention computations.<br/>
👉 **Click** on the **Layer** or **Head** drop-downs to change the model layer or head (zero-indexed).


In [ ]:
from bertviz.transformers_neuron_view import BertModel, BertTokenizer
from bertviz.neuron_view import show

model_type = 'bert'
model_version = 'bert-base-uncased'
model = BertModel.from_pretrained(model_version, output_attentions=True)
tokenizer = BertTokenizer.from_pretrained(model_version, do_lower_case=True)
show(model, model_type, tokenizer, sentence_a, sentence_b, layer=4, head=3)

### b) Investigating **query**, **key**, and **value** patterns

Great! Now that you roughly know how to navigate the neuron-view let's visit some of the questions we have asked before such as: How do the query, key, value calculations look like BERT? What do each of these matrices learn? And are there any patterns given certain inputs? Press on the plus icon next to any of the sentence tokens to see a breakdown!

Remember from the lecture each step to calculate the attention:

- Query **q**: the query vector q encodes the word on the left that is paying attention, i.e. the one that is “querying” the other words. In the example above, the query vector for “on” (the selected word) is highlighted.

- Key **k**: the key vector k encodes the word on the right to which attention is being paid. The key vector and the query vector together determine a compatibility score between the two words.

- **q×k** (elementwise): the elementwise product between the query vector of the selected word and each of the key vectors. This is a precursor to the dot product (the sum of the elementwise product) and is included for visualization purposes because it shows how individual elements in the query and key vectors contribute to the dot product.

- **q·k**: the scaled dot product (see above) of the selected query vector and each of the key vectors. This is the unnormalized attention score.

- **softmax**: the softmax of the scaled dot product. This normalizes the attention scores to be positive and sum to one.

#### Delimeter Patterns

One of the simplest question we can ask is "How is BERT's separator [SEP] token encoded by these neurons?". To answer the question look at the following example and hover over the **[SEP] token**.

In [ ]:
delimeter_sentence_a = "I went to the store."
delimeter_sentence_b = "At the store, I bought fresh strawberries."

show(model, model_type, tokenizer, delimeter_sentence_a, delimeter_sentence_b, layer=4, head=3)

**Q: For the example above, fix the head to be number 3 and look at the "all" sentences option. Look through layers 4-9. What do you notice? Look at the other layers? Does the pattern you noticed still exist?**

*A: TODO - your answer here!*

<details>
  <summary>SOLUTION - click me!</summary>
  
  > All tokens from the left hand-side seem to attend to the [SEP] tokens for layers 4-9. This is not the case for other layers.
</details>

So, how exactly is BERT able to create this pattern? Let’s see if the visualization can provide some clues. Choose layer 7 and head number 3. Hover over the "the" token on the left hand-side and press on the plus icon.

**Q: What do you notice about the *key column* when you hover over "the"? And consequently, what do you notice when you look at the *element-wise multiplication column*?**

*A: TODO - your answer here!*

<details>
  <summary>SOLUTION - click me!</summary>
  
  > In the key column: the key vectors for the two occurrences of [SEP] carry a distinctive signature. They both have a small number of active neurons with strongly positive (blue) or negative (orange) values. A larger number of neurons with values close to zero (light blue/orange or white).
  
  > In the element-wise product column: query vectors tend to match the [SEP] key vectors along those active neurons, resulting in high values for the elementwise product q×k, as in this example.
</details>

For more neuron patterns, we encourage you to read up on the [second blog post](https://towardsdatascience.com/deconstructing-bert-part-2-visualizing-the-inner-workings-of-attention-60a16d86b5c1) made by Jesse Vig!

## TESTING ON FRENCH

In [ ]:
# 2) Load model and tokenizer
model_version = 'bert-base-multilingual-cased'
model = BertModel.from_pretrained(model_version, output_attentions=True)
tokenizer = BertTokenizer.from_pretrained(model_version)

In [ ]:
delimeter_sentence_a = "I went to the store."
delimeter_sentence_b = "At the store, I bought fresh strawberries."

show(model, model_type, tokenizer, delimeter_sentence_a, delimeter_sentence_b, layer=4, head=3)

In [ ]:
# co-référence ? 'il' pointe vers ? jusqu'au  layer 8 par top..
delimeter_sentence_a = "Le président a choisi un premier ministre de droite."
delimeter_sentence_b = "Il a choisi d'ignorer le résultat des élections."

show(model, model_type, tokenizer, delimeter_sentence_a, delimeter_sentence_b, layer=4, head=3)

In [ ]:
# 'Il' pointe vers quoi, chocolat : café ?
# vers 'le café' sur la fin des layers pour cette tête
delimeter_sentence_a = "Le café est un meilleur médicament que le chocolat."
delimeter_sentence_b = "Il guérit le mal de tête."

show(model, model_type, tokenizer, delimeter_sentence_a, delimeter_sentence_b, layer=4, head=3)

## 2) Head View Exploration

The head view visualizes attention in one or more heads from a single Transformer layer.

### a) Instructions on how to use the head view

Each line shows the attention from one token (left) to another (right). Line weight reflects the attention value (ranges from 0 to 1), while line color identifies the attention head. When multiple heads are selected (indicated by the colored tiles at the top), the corresponding  visualizations are overlaid onto one another.

👉 **Hover** over any **token** on the left/right side of the visualization to filter attention from/to that token. <br/>
👉 **Double-click** on any of the **colored tiles** at the top to filter to the corresponding attention head.<br/>
👉 **Single-click** on any of the **colored tiles** to toggle selection of the corresponding attention head. <br/>
👉 **Click** on the **Layer** drop-down to change the model layer (zero-indexed).


In [ ]:
head_view(attention, tokens, sentence_b_start)

### b) Use case: coreference resolution & detecting model bias

The attention-head view allows us to see how certain words in the context relate to each other. Certain heads in the GPT-2 model are shown to do [coreference resolution](https://en.wikipedia.org/wiki/Coreference). You can imagine putting the most attention of pronouns to:
- **gender-specific terms:** she -> the girl, they -> the person
- **names:** she -> Alice, he -> Bob

While these biases may seem harmless, there can be other things to link such as occupation. In this case, ideally the model should attend to gendered items equally. We will show that this is not the case.

Take as an example the following sentences:
- The doctor asked the nurse a question. She ...
- The doctor asked the nurse a question. He ...

Given that the GPT-2 model is an autoregressive model, you can imagine the model to have seen only a portion of the sentence. Which occupation will these pronouns attend to the most?

In [ ]:
from transformers import AutoTokenizer, AutoModel

gpt2_tokenizer = AutoTokenizer.from_pretrained("gpt2")
gpt2_model = AutoModel.from_pretrained("gpt2", output_attentions=True)

In [ ]:
she_ending_sentence = "The doctor asked the nurse a question. She"
he_ending_sentence = "The doctor asked the nurse a question. He"

In [ ]:
she_inputs = gpt2_tokenizer.encode(she_ending_sentence, return_tensors='pt')
she_outputs = gpt2_model(she_inputs)
she_attention = she_outputs[-1]  # Output includes attention weights when output_attentions=True
she_tokens = gpt2_tokenizer.convert_ids_to_tokens(she_inputs[0])

head_view(she_attention, she_tokens, layer=5, heads=[10])

**Q: It has been shown that for gender-specific terms and names layer 5 head 10 of GPT-2 does coreference resolution (as stated in [BertViz's paper](https://aclanthology.org/P19-3007.pdf) and shown in Fig. 4, but more thoroughly studied by [this paper on gender bias](https://aclanthology.org/N18-2003/)). For the same setting, hover over the token "She" on the left handside. Which token is attended to the most by "She" in this particular head? Is it the same case if you start adding other heads (by clicking on the different colors)?**

*A: TODO - your answer here!*

<details>
  <summary>SOLUTION - click me!</summary>
  
  > The "She" token has the most attention to nurse. This shows that the same head that does coreference resolution for names and gender-specific terms also does for occupation but with a certain bias.
  
  > The more heads we add the less clear it gets as to what the model is resolving the ambiguous pronoun "She". However, there is still enough attention for "nurse" in this particular layer 5 with several other attention heads.
</details>

In [ ]:
he_inputs = gpt2_tokenizer.encode(he_ending_sentence, return_tensors='pt')
he_outputs = gpt2_model(he_inputs)
he_attention = he_outputs[-1]  # Output includes attention weights when output_attentions=True
he_tokens = gpt2_tokenizer.convert_ids_to_tokens(he_inputs[0])

head_view(he_attention, he_tokens, layer=5, heads=[10])

**Q: For the same setting, hover over the token "He" to see what the attention head would resolve it to? Does it still have bias? What do you think should be the ideal scenario in such a coreference resolving attention head?**

*A: TODO - your answer here!*

<details>
  <summary>SOLUTION - click me!</summary>
  
  > The "He" token has the most attention to the NP "the doctor". The gender bias towards occupations still exists.
  
  > Ideally the attention should be roughly equal for the different occupations as it is an ambigious situation and there isn't enough context to decide on the occupation.
</details>

### c) Takeaway

**Similar to the conclusion we have made in the prior section, we see that we could modify whole attention heads, in addition to specific neurons, to control the type of bias exhibited by Transformer language models.**

## 3) Model View Exploration

The model view provides a birds-eye view of attention throughout the entire model.

### a) Instructions on how to use the model view
 Each cell shows the attention weights for a particular head, indexed by **layer (row) and head (column)**.  The lines in each cell represent the attention from one token (left) to another (right), with line weight proportional to the attention value (ranges from 0 to 1).

👉 **Click** on any **cell** for a detailed view of attention for the associated attention head (or to unselect that cell). <br/>
👉 Then **hover** over any **token** on the left side of detail view to filter the attention from that token.

In [ ]:
model_view(attention, tokens, sentence_b_start)
# NOTE: reminder that sentence_a = "The cat sat on the mat." and sentence_b = "The cat lay on the rug."

### b) Use case: quickly locating relevant attention-heads

One advantage of the complete model view is that we don't have to try all head and layer combinations to see if certain patterns repeat and can be grouped across different heads. For example, we have previously looked at layer 5, head 10 for coreference resolution. However, if you have tried other layers for the same head you may have noticed that other heads also do similar coreference resolution roles. The same can apply to the example we have seen in the neuron view on the delimiter's role and past-context attending neurons. Let's take a look at how we can find a group of heads that do **paraphrase detection**.

#### Paraphrase detection

First-off, think about what type of an attention pattern paraphrase detecting heads should have.

Given the following sentences:

- "The cat sat on the mat."
- "The cat lay on the rug."

**Q: How do you think each token in one sentence should relate to each other?**

*A: TODO - your answer here!*

<details>
  <summary>SOLUTION - click me!</summary>
  
  > We can expect different constitutents depicting the same meaning such as "lay on" and "sat on" or "the math" and "the rug" to have high attention to each other.

  > This will create a cross-hatch looking pattern.
</details>

**Q: Now that you know what type of patterns the paragraph detecting heads should have, can you notice in the complete birds-eye view which layer/head combinations could be responsible for it?**

*A: TODO - your answer here!*

<details>
  <summary>SOLUTION - click me!</summary>
  
  Possible answers:
  - Layer 0, Head 6
  - Layer 0, Head 9
  - Layer 1, Head 11
  - Layer 3, Head 0

</details>

### c) Takeaway

**We saw a way to use the overall model attention view. What you can next do is test pairs of sentences that shouldn't be paraphrases of each other and see what happens then. For the listed layer/head combinations, are the cross hatches still active, or are they toned down? If the former, it's likely that the noted heads are not encoding paraphrasing related functions.**

## Conclusion

Self-attention is a complex mechanism that gets even harder to interpret when multiple head attentions are combined together. To disentangle this, the BertViz package allows us to look at individual heads in specific layers, and even shows neuron-level visualizations. Interpretability is however not always this easy. In this practical session, most of the role-specific heads were given, and somebody had to visually notice them by testing the heads with different examples. As this is quite tedious, it's good to note that most of these visualizations are often more useful to understand how the attention mechanism works rather than to do in-detail interpretability studies.

## What else can I do?

Visit the [BertViz github repository](https://github.com/jessevig/bertviz) for full documentation and additional use cases. Check out the [aforementioned blog post](https://towardsdatascience.com/deconstructing-bert-part-2-visualizing-the-inner-workings-of-attention-60a16d86b5c1) for a deep dive on BertViz and the attention mechanism.